# 🖊️ Signature Detection & Classification using YOLOv8

This notebook walks through the complete pipeline for building a **Genuine vs. Forged** signature classifier using Ultralytics YOLOv8.

### Steps
1. Environment Setup & Imports
2. Dataset Exploration
3. Dataset Preparation (Train / Val / Test split)
4. Model Training
5. Training Results Visualization
6. Model Evaluation
7. Inference on Test Images
8. Export the Trained Model

---
## Step 1 — Environment Setup & Imports
Install the required packages and import them.

In [ ]:
# Install ultralytics (includes YOLOv8)
!pip install ultralytics --quiet

In [ ]:
import os
import shutil
import random
import glob
from pathlib import Path

import matplotlib.pyplot as plt
import matplotlib.image as mpimg
import numpy as np
from PIL import Image

from ultralytics import YOLO

print("All imports successful! ✅")

---
## Step 2 — Dataset Exploration
Let's explore the raw dataset to understand its structure and size.

In [ ]:
# === CONFIGURE YOUR PATHS HERE ===
BASE_DIR = r"d:\signature_detection_&_verifaction\signatures"
ORG_DIR  = os.path.join(BASE_DIR, "full_org")   # Genuine signatures
FORG_DIR = os.path.join(BASE_DIR, "full_forg")   # Forged signatures

# Output directory for the prepared dataset
DATASET_DIR = r"d:\signature_detection_&_verifaction\datasets\signature_data"

# Count files
org_files  = [f for f in os.listdir(ORG_DIR)  if f.lower().endswith('.png')]
forg_files = [f for f in os.listdir(FORG_DIR) if f.lower().endswith('.png')]

print(f"📂 Genuine signatures (full_org) : {len(org_files)} images")
print(f"📂 Forged  signatures (full_forg): {len(forg_files)} images")
print(f"📊 Total images                  : {len(org_files) + len(forg_files)}")

In [ ]:
# Visualize a few sample images from each class
fig, axes = plt.subplots(2, 5, figsize=(20, 8))
fig.suptitle("Sample Signatures", fontsize=16, fontweight='bold')

# Show 5 genuine samples
for i, ax in enumerate(axes[0]):
    img_path = os.path.join(ORG_DIR, org_files[i])
    img = mpimg.imread(img_path)
    ax.imshow(img, cmap='gray')
    ax.set_title(f"Genuine #{i+1}", color='green', fontweight='bold')
    ax.axis('off')

# Show 5 forged samples
for i, ax in enumerate(axes[1]):
    img_path = os.path.join(FORG_DIR, forg_files[i])
    img = mpimg.imread(img_path)
    ax.imshow(img, cmap='gray')
    ax.set_title(f"Forged #{i+1}", color='red', fontweight='bold')
    ax.axis('off')

plt.tight_layout()
plt.show()

In [ ]:
# Check image dimensions for a few samples
print("Sample image dimensions:")
for f in org_files[:3]:
    img = Image.open(os.path.join(ORG_DIR, f))
    print(f"  {f}: {img.size} (W x H), mode={img.mode}")
for f in forg_files[:3]:
    img = Image.open(os.path.join(FORG_DIR, f))
    print(f"  {f}: {img.size} (W x H), mode={img.mode}")

---
## Step 3 — Dataset Preparation
Reorganize images into the YOLOv8 classification directory format:
```
datasets/signature_data/
├── train/
│   ├── genuine/
│   └── forged/
├── val/
│   ├── genuine/
│   └── forged/
└── test/
    ├── genuine/
    └── forged/
```
We use an **80/10/10** split (Train / Validation / Test).

In [ ]:
def prepare_dataset(org_dir, forg_dir, output_dir, seed=42):
    """Split genuine & forged images into train/val/test folders."""
    
    splits  = ["train", "val", "test"]
    classes = ["genuine", "forged"]

    # Create directory structure
    for split in splits:
        for cls in classes:
            os.makedirs(os.path.join(output_dir, split, cls), exist_ok=True)

    # Collect image files
    org_files  = sorted([f for f in os.listdir(org_dir)  if f.lower().endswith('.png')])
    forg_files = sorted([f for f in os.listdir(forg_dir) if f.lower().endswith('.png')])

    # Shuffle
    random.seed(seed)
    random.shuffle(org_files)
    random.shuffle(forg_files)

    def split_list(file_list):
        n = len(file_list)
        train_end = int(n * 0.80)
        val_end   = int(n * 0.90)
        return file_list[:train_end], file_list[train_end:val_end], file_list[val_end:]

    org_train, org_val, org_test    = split_list(org_files)
    forg_train, forg_val, forg_test = split_list(forg_files)

    def copy_files(files, source_dir, split, cls):
        dest = os.path.join(output_dir, split, cls)
        for f in files:
            shutil.copy2(os.path.join(source_dir, f), os.path.join(dest, f))
        print(f"  ✅ {split}/{cls}: {len(files)} images")

    print("Copying files...")
    copy_files(org_train,  org_dir,  "train", "genuine")
    copy_files(org_val,    org_dir,  "val",   "genuine")
    copy_files(org_test,   org_dir,  "test",  "genuine")
    copy_files(forg_train, forg_dir, "train", "forged")
    copy_files(forg_val,   forg_dir, "val",   "forged")
    copy_files(forg_test,  forg_dir, "test",  "forged")

    print(f"\n🎉 Dataset prepared at: {output_dir}")

# Run the preparation
prepare_dataset(ORG_DIR, FORG_DIR, DATASET_DIR)

In [ ]:
# Verify the split counts
print("📊 Dataset Split Summary:")
print("=" * 40)
for split in ["train", "val", "test"]:
    for cls in ["genuine", "forged"]:
        path = os.path.join(DATASET_DIR, split, cls)
        count = len(os.listdir(path))
        print(f"  {split:6s} / {cls:8s} : {count} images")
    print()

---
## Step 4 — Model Training
Load a pre-trained **YOLOv8 Nano Classification** model and fine-tune it on our dataset.

| Parameter | Value | Description |
|-----------|-------|-------------|
| `model`   | `yolov8n-cls.pt` | Nano variant (smallest, fastest) |
| `epochs`  | 20 | Number of training epochs |
| `imgsz`   | 224 | Input image size |
| `batch`   | 32 | Batch size |

> 💡 **Tip:** You can swap `yolov8n-cls.pt` with `yolov8s-cls.pt` (small) or `yolov8m-cls.pt` (medium) for potentially higher accuracy at the cost of longer training time.

In [ ]:
# Load pre-trained YOLOv8 classification model
model = YOLO("yolov8n-cls.pt")

print(f"Model loaded: {model.model_name}")
print("Ready for training! 🚀")

In [ ]:
# Train the model
results = model.train(
    data=DATASET_DIR,
    epochs=20,
    imgsz=224,
    batch=32,
    project=r"d:\signature_detection_&_verifaction\runs",
    name="signature_classifier",
    patience=5,          # Early stopping patience
    optimizer="Adam",
    lr0=0.001,           # Initial learning rate
    verbose=True
)

print("\n✅ Training complete!")

---
## Step 5 — Training Results Visualization
YOLOv8 automatically saves training curves and metrics. Let's visualize them.

In [ ]:
# Find the latest training run directory
train_dir = r"d:\signature_detection_&_verifaction\runs\signature_classifier"

# Display training results image generated by YOLOv8
results_img_path = os.path.join(train_dir, "results.png")
if os.path.exists(results_img_path):
    img = mpimg.imread(results_img_path)
    plt.figure(figsize=(16, 8))
    plt.imshow(img)
    plt.axis('off')
    plt.title("Training Results", fontsize=16, fontweight='bold')
    plt.show()
else:
    print(f"⚠️ results.png not found at {results_img_path}")
    print("  This is normal if training hasn't completed yet.")

In [ ]:
# Display the confusion matrix
confusion_matrix_path = os.path.join(train_dir, "confusion_matrix.png")
if os.path.exists(confusion_matrix_path):
    img = mpimg.imread(confusion_matrix_path)
    plt.figure(figsize=(8, 8))
    plt.imshow(img)
    plt.axis('off')
    plt.title("Confusion Matrix", fontsize=16, fontweight='bold')
    plt.show()
else:
    print(f"⚠️ confusion_matrix.png not found at {confusion_matrix_path}")

---
## Step 6 — Model Evaluation
Evaluate the trained model on the **validation** set to get accuracy metrics.

In [ ]:
# Load the best weights from training
best_model_path = os.path.join(train_dir, "weights", "best.pt")
best_model = YOLO(best_model_path)

print(f"✅ Loaded best model from: {best_model_path}")

In [ ]:
# Validate on the validation set
val_results = best_model.val(
    data=DATASET_DIR,
    imgsz=224,
    batch=32,
    split="val"
)

print(f"\n📊 Validation Results:")
print(f"  Top-1 Accuracy: {val_results.top1:.4f}")
print(f"  Top-5 Accuracy: {val_results.top5:.4f}")

---
## Step 7 — Inference on Test Images
Run the trained model on unseen test images to see predictions.

In [ ]:
# Collect test images
test_genuine = glob.glob(os.path.join(DATASET_DIR, "test", "genuine", "*.png"))
test_forged  = glob.glob(os.path.join(DATASET_DIR, "test", "forged",  "*.png"))

# Pick a few samples from each class
sample_genuine = test_genuine[:4]
sample_forged  = test_forged[:4]
sample_images  = sample_genuine + sample_forged
actual_labels  = ["genuine"] * len(sample_genuine) + ["forged"] * len(sample_forged)

print(f"Running inference on {len(sample_images)} test images...")

In [ ]:
# Run predictions
predictions = best_model.predict(
    source=sample_images,
    imgsz=224,
    verbose=False
)

# Visualize predictions
fig, axes = plt.subplots(2, 4, figsize=(20, 10))
fig.suptitle("Model Predictions on Test Images", fontsize=18, fontweight='bold')

for i, (ax, result) in enumerate(zip(axes.flatten(), predictions)):
    img = Image.open(sample_images[i])
    ax.imshow(img, cmap='gray')
    
    # Get predicted class and confidence
    probs = result.probs
    pred_class_id = probs.top1
    pred_class_name = result.names[pred_class_id]
    confidence = probs.top1conf.item()
    actual = actual_labels[i]
    
    # Color code: green if correct, red if wrong
    is_correct = pred_class_name == actual
    color = 'green' if is_correct else 'red'
    status = '✅' if is_correct else '❌'
    
    ax.set_title(
        f"Pred: {pred_class_name} ({confidence:.1%})\nActual: {actual} {status}",
        color=color, fontweight='bold', fontsize=11
    )
    ax.axis('off')

plt.tight_layout()
plt.show()

In [ ]:
# Calculate test set accuracy
all_test_images = test_genuine + test_forged
all_actual = ["genuine"] * len(test_genuine) + ["forged"] * len(test_forged)

all_preds = best_model.predict(source=all_test_images, imgsz=224, verbose=False)

correct = 0
total = len(all_preds)

for pred, actual in zip(all_preds, all_actual):
    pred_class = pred.names[pred.probs.top1]
    if pred_class == actual:
        correct += 1

accuracy = correct / total
print(f"\n📊 Test Set Results:")
print(f"  Total images : {total}")
print(f"  Correct      : {correct}")
print(f"  Incorrect    : {total - correct}")
print(f"  Accuracy     : {accuracy:.2%}")

---
## Step 8 — Export the Trained Model
Export the model to different formats for deployment.

In [ ]:
# Export to ONNX format (widely supported for deployment)
exported_path = best_model.export(format="onnx", imgsz=224)
print(f"\n✅ Model exported to ONNX: {exported_path}")

In [ ]:
# You can also export to other formats:
# best_model.export(format="torchscript")  # PyTorch TorchScript
# best_model.export(format="tflite")        # TensorFlow Lite (mobile)
# best_model.export(format="engine")        # TensorRT (NVIDIA GPUs)

print("\n🎉 All done! Your signature classifier is trained and ready.")
print(f"\n📁 Best weights saved at:")
print(f"   {best_model_path}")

---
## 🔮 Using the Model for New Predictions
After training, you can load the model and classify any new signature image:

In [ ]:
# ==========================================
# Quick Usage - Classify a new signature
# ==========================================
# 
# from ultralytics import YOLO
# 
# model = YOLO(r"d:\signature_detection_&_verifaction\runs\signature_classifier\weights\best.pt")
# result = model.predict("path/to/new_signature.png", imgsz=224)
# 
# pred_class = result[0].names[result[0].probs.top1]
# confidence = result[0].probs.top1conf.item()
# print(f"Prediction: {pred_class} (confidence: {confidence:.2%})")

print("Uncomment the code above and provide a path to classify a new signature.")